# Lanzamos scraping 7 parroquias 5 subtipos de vivienda en pisos.ad

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from andorra_housing.scraping.parsers import pisos_ad
from andorra_housing.scraping import almacenamiento
import duckdb

In [ ]:
display(pisos_ad.PARROQUIAS_AD)
display(pisos_ad.SUBTIPOS_AD)

pisos_ad.ejecutar()

In [ ]:
conexion = duckdb.connect(database="../data/andorra_housing.duckdb")
df = conexion.execute("SELECT * FROM anuncios").fetchdf()
avisos = conexion.execute("SELECT * FROM avisos").fetchdf()
consulta = conexion.execute("SELECT portal_id, COUNT(*) as count FROM avisos GROUP BY portal_id ORDER BY count DESC").fetchdf()
print(conexion.execute("SELECT parish, COUNT(*) as count, property_type FROM anuncios GROUP BY parish, property_type ORDER BY parish").fetchdf())
print(conexion.execute("SELECT parish, COUNT(*) as count FROM anuncios GROUP BY parish ORDER BY parish").fetchdf())
conexion.close()
display(avisos)
display(df.describe())
display(consulta)

## CONTEO TIPOS DE VIVIENDA X PARROQUIA, Y CONTEO X PARROQUIA

In [ ]:
conexion = duckdb.connect(database="../data/andorra_housing.duckdb")
print(conexion.execute("SELECT parish, COUNT(*) as count, property_type FROM anuncios GROUP BY parish, property_type ORDER BY parish").fetchdf())
print(conexion.execute("SELECT parish, COUNT(*) as count FROM anuncios GROUP BY parish ORDER BY parish").fetchdf())
conexion.close()


## Tipo de discrepancias con el LLM

In [ ]:
conexion = duckdb.connect(database="../data/andorra_housing.duckdb")
print(conexion.execute("SELECT campo ,COUNT(*) as count, SUM(valor_estructurado) as si_scraping, SUM(valor_llm) as si_llm FROM avisos GROUP BY campo ORDER BY count DESC").fetchdf())
a = conexion.execute("SELECT url FROM anuncios WHERE portal_id = 9209 OR portal_id = 9305").fetchdf()
conexion.close()
for _,desc in a["url"].items():
    print(str(desc))

In [ ]:
conexion = duckdb.connect(database="../data/andorra_housing.duckdb")
print(conexion.execute("SELECT * FROM avisos WHERE campo = 'has_balcony' AND valor_estructurado = 1").fetchdf())
revision = conexion.execute("SELECT * FROM avisos WHERE portal_id = 9209 OR portal_id = 9305 OR portal_id = 8889 AND valor_estructurado = True").fetchdf()
conexion.close()

In [ ]:
revision = revision[revision["valor_estructurado"] == True]
display(revision)
for _,desc in revision["descripcion"].items():
    print(str(desc))


El llm dice que si hay terraza, trastero, balcon y ascensor, mientras que no consta en al ficha tecnica

## BUCLE para revision de datos

In [ ]:
from andorra_housing.scraping import almacenamiento

conexion = almacenamiento.conectar()   # esto SÍ añade la columna veredicto si falta
portal = "pisos_ad"

avisos = almacenamiento.obtener_avisos(conexion, portal)
print(f"{len(avisos)} avisos pendientes de revisar para {portal}")

for portal_id, campo, valor_estructurado, valor_llm, descripcion in avisos:
    print(f"--- {portal_id} | {campo} | estructurado={valor_estructurado} LLM={valor_llm} ---")
    print(descripcion)

    veredicto = input("¿Correcto (c) o alucinación (a)? ")
    almacenamiento.guardar_veredicto_aviso(conexion, portal, portal_id, campo, veredicto)
    print()


In [ ]:
from andorra_housing.scraping.parsers import pisos_ad

pisos_ad.reintentar_fallos()


In [ ]:
conexion = almacenamiento.conectar()
conexion.execute("show tables").fetchdf()
fallos = conexion.execute("SELECT * FROM fallos_llm").fetchdf()
conexion.close()

In [ ]:
for _,resp in fallos["respuesta_cruda"].items():
    print(str(resp))

In [ ]:
conexion = almacenamiento.conectar()
condition = conexion.execute("SELECT condition FROM anuncios").fetchdf()
conexion.close()

In [ ]:
condition.value_counts(dropna=False, normalize=True)


In [ ]:
conexion = almacenamiento.conectar()
df = conexion.execute("SELECT * FROM anuncios").fetchdf()
expenses = conexion.execute("SELECT expenses_included, expenses_detail FROM anuncios").fetchdf()
expenses_filtro = conexion.execute("SELECT expenses_included, expenses_detail, descripcion_raw, url FROM anuncios WHERE expenses_included IS NOT NULL AND expenses_included != 'no' AND expenses_detail IS NULL").fetchdf()
conexion.close()

In [ ]:
df["expenses_included"].value_counts(dropna=False, normalize=True)

In [ ]:
df.columns

In [ ]:
df["expenses_included"].value_counts(dropna=False, normalize=True)

In [ ]:
print(expenses_filtro)

In [ ]:
for _,text in expenses_filtro["descripcion_raw"].items():
    print(str(text))

1. no dice nada sobre gastos 
2. no dice nada, pero tiene spa comunitario 
3. 

5. no dice nada 